<a href="https://colab.research.google.com/github/fealmutairi79-0/CS-220P/blob/main/notebooks/02a-building-data-pipelines-in-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Data Pipelines in PyTorch

In [1]:
import pathlib

import numpy as np
import pandas as pd
from sklearn import preprocessing, pipeline
import torch
from torch import utils

In [2]:

print(torch.__version__)

2.9.0+cpu


## Creating a DataLoader from existing tensors

* PyTorch’s `DataLoader` helps efficiently load data in mini-batches.
* Can shuffle data each epoch for better generalization: always set `shuffle=True` when training; can set `shuffle=False` when evaluating.
* Expects a `Dataset` with the following methods:

  1. `__len__(self)` → returns dataset size
  2. `__getitem__(self, index)` → returns sample and target values for the index into the provided `Dataset`.

In [3]:
utils.data.DataLoader?

In [4]:
prng = torch.manual_seed(42)
dataset = torch.rand(10, 8, generator=prng)

In [5]:
print(dataset)

tensor([[0.8823, 0.9150, 0.3829, 0.9593, 0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936, 0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274, 0.2696, 0.4414, 0.2969, 0.8317],
        [0.1053, 0.2695, 0.3588, 0.1994, 0.5472, 0.0062, 0.9516, 0.0753],
        [0.8860, 0.5832, 0.3376, 0.8090, 0.5779, 0.9040, 0.5547, 0.3423],
        [0.6343, 0.3644, 0.7104, 0.9464, 0.7890, 0.2814, 0.7886, 0.5895],
        [0.7539, 0.1952, 0.0050, 0.3068, 0.1165, 0.9103, 0.6440, 0.7071],
        [0.6581, 0.4913, 0.8913, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278],
        [0.6532, 0.3958, 0.9147, 0.2036, 0.2018, 0.2018, 0.9497, 0.6666],
        [0.9811, 0.0874, 0.0041, 0.1088, 0.1637, 0.7025, 0.6790, 0.9155]])


In [6]:
dataloader = utils.data.DataLoader(dataset)

In [7]:
# default batch_size = 1
for i, batch in enumerate(dataloader):
    print(batch)

tensor([[0.8823, 0.9150, 0.3829, 0.9593, 0.3904, 0.6009, 0.2566, 0.7936]])
tensor([[0.9408, 0.1332, 0.9346, 0.5936, 0.8694, 0.5677, 0.7411, 0.4294]])
tensor([[0.8854, 0.5739, 0.2666, 0.6274, 0.2696, 0.4414, 0.2969, 0.8317]])
tensor([[0.1053, 0.2695, 0.3588, 0.1994, 0.5472, 0.0062, 0.9516, 0.0753]])
tensor([[0.8860, 0.5832, 0.3376, 0.8090, 0.5779, 0.9040, 0.5547, 0.3423]])
tensor([[0.6343, 0.3644, 0.7104, 0.9464, 0.7890, 0.2814, 0.7886, 0.5895]])
tensor([[0.7539, 0.1952, 0.0050, 0.3068, 0.1165, 0.9103, 0.6440, 0.7071]])
tensor([[0.6581, 0.4913, 0.8913, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278]])
tensor([[0.6532, 0.3958, 0.9147, 0.2036, 0.2018, 0.2018, 0.9497, 0.6666]])
tensor([[0.9811, 0.0874, 0.0041, 0.1088, 0.1637, 0.7025, 0.6790, 0.9155]])


In [8]:
# always manually set the batch_size for your problem!
dataloader = utils.data.DataLoader(dataset, batch_size=3)

for i, batch in enumerate(dataloader):
    print(batch)

tensor([[0.8823, 0.9150, 0.3829, 0.9593, 0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936, 0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274, 0.2696, 0.4414, 0.2969, 0.8317]])
tensor([[0.1053, 0.2695, 0.3588, 0.1994, 0.5472, 0.0062, 0.9516, 0.0753],
        [0.8860, 0.5832, 0.3376, 0.8090, 0.5779, 0.9040, 0.5547, 0.3423],
        [0.6343, 0.3644, 0.7104, 0.9464, 0.7890, 0.2814, 0.7886, 0.5895]])
tensor([[0.7539, 0.1952, 0.0050, 0.3068, 0.1165, 0.9103, 0.6440, 0.7071],
        [0.6581, 0.4913, 0.8913, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278],
        [0.6532, 0.3958, 0.9147, 0.2036, 0.2018, 0.2018, 0.9497, 0.6666]])
tensor([[0.9811, 0.0874, 0.0041, 0.1088, 0.1637, 0.7025, 0.6790, 0.9155]])


In [9]:
# drop_last=True useful when dataset is not evenly divisible by the batch_size
dataloader = utils.data.DataLoader(dataset, batch_size=3, drop_last=True)

for i, batch in enumerate(dataloader):
    print(batch)

tensor([[0.8823, 0.9150, 0.3829, 0.9593, 0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936, 0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274, 0.2696, 0.4414, 0.2969, 0.8317]])
tensor([[0.1053, 0.2695, 0.3588, 0.1994, 0.5472, 0.0062, 0.9516, 0.0753],
        [0.8860, 0.5832, 0.3376, 0.8090, 0.5779, 0.9040, 0.5547, 0.3423],
        [0.6343, 0.3644, 0.7104, 0.9464, 0.7890, 0.2814, 0.7886, 0.5895]])
tensor([[0.7539, 0.1952, 0.0050, 0.3068, 0.1165, 0.9103, 0.6440, 0.7071],
        [0.6581, 0.4913, 0.8913, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278],
        [0.6532, 0.3958, 0.9147, 0.2036, 0.2018, 0.2018, 0.9497, 0.6666]])


## Combining two tensors into a joint dataset

Often we will have datasets that combine two (or more!) tensors and we want to be able to shuffle and grab batches of all the different tensors and retrieve the results as tuples.

In [10]:
class JointDataset(utils.data.Dataset):
    """Example of a creating a custom dataset."""

    def __init__(self, d1, d2):
        self. _d1 = d1
        self._d2 = d2

    def __len__(self):
        return len(self._d1)

    def __getitem__(self, idx):
        return self._d1[idx], self._d2[idx]



In [11]:
features = torch.rand(10, 8, generator=prng)
target = torch.rand(10, 1, generator=prng)

In [12]:
dataset = JointDataset(features, target)

In [13]:
dataloader = utils.data.DataLoader(dataset, batch_size=3)

for i, (feature_batch, target_batch) in enumerate(dataloader):
    print(feature_batch, target_batch)

tensor([[0.7860, 0.1115, 0.2477, 0.6524, 0.6057, 0.3725, 0.7980, 0.8399],
        [0.1374, 0.2331, 0.9578, 0.3313, 0.3227, 0.0162, 0.2137, 0.6249],
        [0.4340, 0.1371, 0.5117, 0.1585, 0.0758, 0.2247, 0.0624, 0.1816]]) tensor([[0.2709],
        [0.9295],
        [0.6115]])
tensor([[0.9998, 0.5944, 0.6541, 0.0337, 0.1716, 0.3336, 0.5782, 0.0600],
        [0.2846, 0.2007, 0.5014, 0.3139, 0.4654, 0.1612, 0.1568, 0.2083],
        [0.3289, 0.1054, 0.9192, 0.4008, 0.9302, 0.6558, 0.0766, 0.8460]]) tensor([[0.2234],
        [0.2469],
        [0.4761]])
tensor([[0.3624, 0.3083, 0.0850, 0.0029, 0.6431, 0.3908, 0.6947, 0.0897],
        [0.8712, 0.1330, 0.4137, 0.6044, 0.7581, 0.9037, 0.9555, 0.1035],
        [0.6258, 0.2849, 0.4452, 0.1258, 0.9554, 0.1330, 0.7672, 0.6757]]) tensor([[0.7792],
        [0.3722],
        [0.2147]])
tensor([[0.6625, 0.2297, 0.9545, 0.6099, 0.5643, 0.0594, 0.7099, 0.4250]]) tensor([[0.3288]])


In [14]:
# this use-case is so common that their is a built-in class suporting it!
utils.data.TensorDataset?

In [15]:
dataset = utils.data.TensorDataset(features, target)#بنستخدمه في المشروع
dataloader = utils.data.DataLoader(dataset, batch_size=3)

for i, (feature_batch, target_batch) in enumerate(dataloader):
    print(feature_batch, target_batch)

tensor([[0.7860, 0.1115, 0.2477, 0.6524, 0.6057, 0.3725, 0.7980, 0.8399],
        [0.1374, 0.2331, 0.9578, 0.3313, 0.3227, 0.0162, 0.2137, 0.6249],
        [0.4340, 0.1371, 0.5117, 0.1585, 0.0758, 0.2247, 0.0624, 0.1816]]) tensor([[0.2709],
        [0.9295],
        [0.6115]])
tensor([[0.9998, 0.5944, 0.6541, 0.0337, 0.1716, 0.3336, 0.5782, 0.0600],
        [0.2846, 0.2007, 0.5014, 0.3139, 0.4654, 0.1612, 0.1568, 0.2083],
        [0.3289, 0.1054, 0.9192, 0.4008, 0.9302, 0.6558, 0.0766, 0.8460]]) tensor([[0.2234],
        [0.2469],
        [0.4761]])
tensor([[0.3624, 0.3083, 0.0850, 0.0029, 0.6431, 0.3908, 0.6947, 0.0897],
        [0.8712, 0.1330, 0.4137, 0.6044, 0.7581, 0.9037, 0.9555, 0.1035],
        [0.6258, 0.2849, 0.4452, 0.1258, 0.9554, 0.1330, 0.7672, 0.6757]]) tensor([[0.7792],
        [0.3722],
        [0.2147]])
tensor([[0.6625, 0.2297, 0.9545, 0.6099, 0.5643, 0.0594, 0.7099, 0.4250]]) tensor([[0.3288]])


## Shuffle, batch, repeat

A key aspect of training neural networks effectively using stochastic gradient descent is to repeated sample batches of data from the shuffled dataset.

In [16]:
dataset = utils.data.TensorDataset(features, target)
train_dataloader = utils.data.DataLoader(
    dataset,
    batch_size=3,
    shuffle=True,   # always shuffle during training!
)


In [17]:
for i, (feature_batch, target_batch) in enumerate(train_dataloader):
    print(feature_batch, target_batch)

tensor([[0.4340, 0.1371, 0.5117, 0.1585, 0.0758, 0.2247, 0.0624, 0.1816],
        [0.2846, 0.2007, 0.5014, 0.3139, 0.4654, 0.1612, 0.1568, 0.2083],
        [0.3624, 0.3083, 0.0850, 0.0029, 0.6431, 0.3908, 0.6947, 0.0897]]) tensor([[0.6115],
        [0.2469],
        [0.7792]])
tensor([[0.1374, 0.2331, 0.9578, 0.3313, 0.3227, 0.0162, 0.2137, 0.6249],
        [0.7860, 0.1115, 0.2477, 0.6524, 0.6057, 0.3725, 0.7980, 0.8399],
        [0.3289, 0.1054, 0.9192, 0.4008, 0.9302, 0.6558, 0.0766, 0.8460]]) tensor([[0.9295],
        [0.2709],
        [0.4761]])
tensor([[0.8712, 0.1330, 0.4137, 0.6044, 0.7581, 0.9037, 0.9555, 0.1035],
        [0.9998, 0.5944, 0.6541, 0.0337, 0.1716, 0.3336, 0.5782, 0.0600],
        [0.6258, 0.2849, 0.4452, 0.1258, 0.9554, 0.1330, 0.7672, 0.6757]]) tensor([[0.3722],
        [0.2234],
        [0.2147]])
tensor([[0.6625, 0.2297, 0.9545, 0.6099, 0.5643, 0.0594, 0.7099, 0.4250]]) tensor([[0.3288]])


In [18]:
epochs = 2
for epoch in range(epochs):
    for i, (X, y) in enumerate(train_dataloader):
        print(f"Epoch {epoch}: features: {X}, target: {y}")

Epoch 0: features: tensor([[0.1374, 0.2331, 0.9578, 0.3313, 0.3227, 0.0162, 0.2137, 0.6249],
        [0.3624, 0.3083, 0.0850, 0.0029, 0.6431, 0.3908, 0.6947, 0.0897],
        [0.9998, 0.5944, 0.6541, 0.0337, 0.1716, 0.3336, 0.5782, 0.0600]]), target: tensor([[0.9295],
        [0.7792],
        [0.2234]])
Epoch 0: features: tensor([[0.2846, 0.2007, 0.5014, 0.3139, 0.4654, 0.1612, 0.1568, 0.2083],
        [0.8712, 0.1330, 0.4137, 0.6044, 0.7581, 0.9037, 0.9555, 0.1035],
        [0.6258, 0.2849, 0.4452, 0.1258, 0.9554, 0.1330, 0.7672, 0.6757]]), target: tensor([[0.2469],
        [0.3722],
        [0.2147]])
Epoch 0: features: tensor([[0.3289, 0.1054, 0.9192, 0.4008, 0.9302, 0.6558, 0.0766, 0.8460],
        [0.7860, 0.1115, 0.2477, 0.6524, 0.6057, 0.3725, 0.7980, 0.8399],
        [0.4340, 0.1371, 0.5117, 0.1585, 0.0758, 0.2247, 0.0624, 0.1816]]), target: tensor([[0.4761],
        [0.2709],
        [0.6115]])
Epoch 0: features: tensor([[0.6625, 0.2297, 0.9545, 0.6099, 0.5643, 0.0594, 0.7099

## Creating a dataset from files on your local disk

In [19]:
%%bash

ls -l ./sample_data

total 55504
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json
-rw-r--r-- 1 root root   301141 Jan 16 14:24 california_housing_test.csv
-rw-r--r-- 1 root root  1706430 Jan 16 14:24 california_housing_train.csv
-rw-r--r-- 1 root root 18289443 Jan 16 14:24 mnist_test.csv
-rw-r--r-- 1 root root 36523880 Jan 16 14:24 mnist_train_small.csv
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md


### Loading files into DataFrames

In [20]:
DATA_DIR = pathlib.Path("./sample_data")

In [21]:
housing_train_df = pd.read_csv(DATA_DIR / "california_housing_train.csv")
housing_test_df = pd.read_csv(DATA_DIR / "california_housing_test.csv")


In [22]:
housing_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           17000 non-null  float64
 1   latitude            17000 non-null  float64
 2   housing_median_age  17000 non-null  float64
 3   total_rooms         17000 non-null  float64
 4   total_bedrooms      17000 non-null  float64
 5   population          17000 non-null  float64
 6   households          17000 non-null  float64
 7   median_income       17000 non-null  float64
 8   median_house_value  17000 non-null  float64
dtypes: float64(9)
memory usage: 1.2 MB


In [23]:
housing_train_features_df = housing_train_df.drop("median_house_value", axis=1)
housing_train_target = housing_train_df.loc[:, "median_house_value"]

housing_test_features_df = housing_test_df.drop("median_house_value", axis=1)
housing_test_target = housing_test_df.loc[:, "median_house_value"]

### Converting from DataFrames to Tensors

In [24]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


def dataframe_to_tensor(df, dtype=torch.float32):
    arr = df.to_numpy()
    return array_to_tensor(arr, dtype)


def series_to_tensor(s, dtype=torch.float32):
    arr = s.to_numpy()
    return array_to_tensor(arr, dtype)


In [25]:
prepare_housing_features = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_housing_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor
    ),
    preprocessing.FunctionTransformer(
        func=torch.unsqueeze,
        kw_args={
            "dim": 1
        }
    )
)



In [26]:
X_train = prepare_housing_features.fit_transform(housing_train_features_df)
y_train = prepare_housing_target.fit_transform(housing_train_target)
housing_train_dataset = utils.data.TensorDataset(
    X_train,
    y_train,
)

X_test = prepare_housing_features.transform(housing_test_features_df)
y_test = prepare_housing_target.transform(housing_test_target)
housing_test_dataset = utils.data.TensorDataset(
    X_test,
    y_test,
)

In [27]:
housing_train_dataloader = utils.data.DataLoader(
    housing_train_dataset,
    batch_size=4,
    shuffle=True
)

for i, (X, y) in enumerate(housing_train_dataloader):
    print(X, y)
    if i > 4:
        break

tensor([[ 5.4965e-01, -7.6510e-01,  6.6822e-01, -3.7418e-01, -1.8841e-01,
         -3.0891e-01, -1.2541e-01,  2.3763e-01],
        [ 9.5861e-01, -7.2767e-01,  7.4767e-01, -8.8842e-02,  1.0888e-02,
          3.4276e-01,  9.0448e-02, -5.4849e-01],
        [-5.5752e-01, -6.7948e-02, -1.0797e+00, -1.6361e-02, -3.4190e-02,
         -6.8455e-02,  2.0235e-03, -5.7847e-01],
        [ 5.7957e-01, -8.0721e-01,  7.4767e-01,  1.0291e-01, -1.7417e-01,
         -2.8887e-01, -1.6442e-01,  2.1080e+00]]) tensor([[360600.],
        [111200.],
        [205800.],
        [477100.]])
tensor([[-0.8767,  1.0971,  1.1449, -0.0875, -0.1172, -0.2932, -0.1644, -0.5099],
        [ 0.6793, -0.7838,  0.9860, -0.9013, -0.8432, -0.5590, -0.8224, -0.9135],
        [-0.1037,  0.5731, -1.6358, -0.5159, -0.7246, -0.6260, -0.6586,  1.1038],
        [-1.1111,  1.4293, -0.3646,  0.5277,  0.7321,  0.6930,  0.9435, -0.5643]]) tensor([[ 88800.],
        [165300.],
        [123600.],
        [ 95300.]])
tensor([[-1.2109,  0.830

In [28]:
# TODO: Add example of loading data directly from disk to mimic dataset that doesn't fit in memory!